In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
emp_data = [(1,'manish',50000,'IT','m'),
(2,'vikash',60000,'sales','m'),
(3,'raushan',70000,'marketing','m'),
(4,'mukesh',80000,'IT','m'),
(5,'priti',90000,'sales','f'),
(6,'nikita',45000,'marketing','f'),
(7,'ragini',55000,'marketing','f'),
(8,'rashi',100000,'IT','f'),
(9,'aditya',65000,'IT','m'),
(10,'rahul',50000,'marketing','m'),
(11,'rakhi',50000,'IT','f'),
(12,'akhilesh',90000,'sales','m')]

emp_schema = ['id','name','salary','dept','gender']
emp_df = spark.createDataFrame(data=emp_data,schema=emp_schema)

#### What is the % of sales each month based on last 6 months sales.

In [0]:
product_data = [
(1,"iphone","01-01-2023",1500000),
(2,"samsung","01-01-2023",1100000),
(3,"oneplus","01-01-2023",1100000),
(1,"iphone","01-02-2023",1300000),
(2,"samsung","01-02-2023",1120000),
(3,"oneplus","01-02-2023",1120000),
(1,"iphone","01-03-2023",1600000),
(2,"samsung","01-03-2023",1080000),
(3,"oneplus","01-03-2023",1160000),
(1,"iphone","01-04-2023",1700000),
(2,"samsung","01-04-2023",1800000),
(3,"oneplus","01-04-2023",1170000),
(1,"iphone","01-05-2023",1200000),
(2,"samsung","01-05-2023",980000),
(3,"oneplus","01-05-2023",1175000),
(1,"iphone","01-06-2023",1100000),
(2,"samsung","01-06-2023",1100000),
(3,"oneplus","01-06-2023",1200000)
]
product_schema=['product_id','product_name','sales_date','sales']
product_df=spark.createDataFrame(data=product_data,schema=product_schema)

emp_data = [(1,'manish',50000,'IT','m'),
(2,'vikash',60000,'sales','m'),
(3,'raushan',70000,'marketing','m'),
(4,'mukesh',80000,'IT','m'),
(5,'priti',90000,'sales','f'),
(6,'nikita',45000,'marketing','f'),
(7,'ragini',55000,'marketing','f'),
(8,'rashi',100000,'IT','f'),
(9,'aditya',65000,'IT','m'),
(10,'rahul',50000,'marketing','m'),
(11,'rakhi',50000,'IT','f'),
(12,'akhilesh',90000,'sales','m')]

emp_schema=['id','name','salary','dept','gender']

emp_df = spark.createDataFrame(data=emp_data,schema=emp_schema)
emp_df = emp_df.select('id','name','salary','gender','dept')
emp_df.show()

+---+--------+------+------+---------+
| id|    name|salary|gender|     dept|
+---+--------+------+------+---------+
|  1|  manish| 50000|     m|       IT|
|  2|  vikash| 60000|     m|    sales|
|  3| raushan| 70000|     m|marketing|
|  4|  mukesh| 80000|     m|       IT|
|  5|   priti| 90000|     f|    sales|
|  6|  nikita| 45000|     f|marketing|
|  7|  ragini| 55000|     f|marketing|
|  8|   rashi|100000|     f|       IT|
|  9|  aditya| 65000|     m|       IT|
| 10|   rahul| 50000|     m|marketing|
| 11|   rakhi| 50000|     f|       IT|
| 12|akhilesh| 90000|     m|    sales|
+---+--------+------+------+---------+



In [0]:
window=Window.partitionBy("product_id").orderBy("sales_date")
last_month_df=product_df.withColumn("previous_month_sales",lag(col("sales"),1).over(window))

last_month_df.show()

+----------+------------+----------+-------+--------------------+
|product_id|product_name|sales_date|  sales|previous_month_sales|
+----------+------------+----------+-------+--------------------+
|         1|      iphone|01-01-2023|1500000|                NULL|
|         1|      iphone|01-02-2023|1300000|             1500000|
|         1|      iphone|01-03-2023|1600000|             1300000|
|         1|      iphone|01-04-2023|1700000|             1600000|
|         1|      iphone|01-05-2023|1200000|             1700000|
|         1|      iphone|01-06-2023|1100000|             1200000|
|         2|     samsung|01-01-2023|1100000|                NULL|
|         2|     samsung|01-02-2023|1120000|             1100000|
|         2|     samsung|01-03-2023|1080000|             1120000|
|         2|     samsung|01-04-2023|1800000|             1080000|
|         2|     samsung|01-05-2023| 980000|             1800000|
|         2|     samsung|01-06-2023|1100000|              980000|
|         

#### What is the % of loss or gain based on previous month sales?

In [0]:
last_month_df.withColumn("per_loss_gain",
                         round(((col("sales")-col("previous_month_sales"))/col("sales"))*100,2))\
                             .show()

+----------+------------+----------+-------+--------------------+-------------+
|product_id|product_name|sales_date|  sales|previous_month_sales|per_loss_gain|
+----------+------------+----------+-------+--------------------+-------------+
|         1|      iphone|01-01-2023|1500000|                NULL|         NULL|
|         1|      iphone|01-02-2023|1300000|             1500000|       -15.38|
|         1|      iphone|01-03-2023|1600000|             1300000|        18.75|
|         1|      iphone|01-04-2023|1700000|             1600000|         5.88|
|         1|      iphone|01-05-2023|1200000|             1700000|       -41.67|
|         1|      iphone|01-06-2023|1100000|             1200000|        -9.09|
|         2|     samsung|01-01-2023|1100000|                NULL|         NULL|
|         2|     samsung|01-02-2023|1120000|             1100000|         1.79|
|         2|     samsung|01-03-2023|1080000|             1120000|         -3.7|
|         2|     samsung|01-04-2023|1800